<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.3: 常见传递模式

**上一节: [FIRRTL AST Traversal](4.2_firrtl_ast_traversal.ipynb)**<br>
**下一节: [A FIRRTL Transform 示例](4.4_firrtl_add_ops_per_module.ipynb)**

### 添加语句
假设我们想编写一个传递(pass)来拆分嵌套的DoPrim表达式，从而将以下代码：
```
电路 Top:
  模块 Top :
  输入 x: UInt<3>
  输入 y: UInt<3>
  输入 z: UInt<3>
  输出 o: UInt<3>
  o <= add(x, add(y, z))
```
转换为：
```
电路 Top:
  模块 Top :
  输入 x: UInt<3>
  输入 y: UInt<3>
  输入 z: UInt<3>
  输出 o: UInt<3>
  node GEN_1 = add(y, z)
  o <= add(x, GEN_1)
```

我们首先需要遍历AST到每个语句和表达式。然后，当我们看到一个DoPrim时，我们需要向模块的主体添加一个新的DefNode，并在DoPrim的位置插入对该DefNode的引用。下面的代码实现了这一点（并保留了Info令牌）。注意`Namespace`是一个位于[Namespace.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/Namespace.scala)的实用函数。

```scala
对象 Splitter extends Pass {
  def name = "Splitter!"
  /** 在每个模块上运行splitM **/
  def run(c: 电路): 电路 = c.copy(modules = c.modules map(splitM(_)))

  /** 在每个模块的主体上运行splitS **/
  def splitM(m: DefModule): DefModule = m map splitS(Namespace(m))

  /** 在所有子表达式上运行splitE。
    * 如果语句包含额外语句，返回一个包含它们和 
    *    新语句的Block；否则，返回新语句。*/
  def splitS(namespace: Namespace)(s: Statement): Statement = {
    val block = mutable.ArrayBuffer[Statement]()
    s match {
      case s: HasInfo => 
        val newStmt = s map splitE(block, namespace, s.info)
        block.length match {
          case 0 => newStmt
          case _ => Block(block.toSeq :+ newStmt)
        }
      case s => s map splitS(namespace)
  }

  /** 在所有子表达式上运行splitE。
    * 如果e是DoPrim，向block添加一个新的DefNode并返回对
    * DefNode的引用；否则返回e。*/
  def splitE(block: mutable.ArrayBuffer[Statement], namespace: Namespace, 
             info: Info)(e: Expression): Expression = e map splitE(block, namespace, info) match {
    case e: DoPrim =>
      val newName = namespace.newTemp
      block += DefNode(info, newName, e)
      Ref(newName, e.tpe)
    case _ => e
  }
}
```
### 删除语句
假设我们想编写一个传递来内联所有值为字面量的DefNodes，从而将以下代码：
```
电路 Top:
  模块 Top :
  输入 x: UInt<3>
  输出 o: UInt<4>
  node y = UInt(1)
  o <= add(x, y)
```
转换为：
```
电路 Top:
  模块 Top :
  输入 x: UInt<3>
  输出 y: UInt<4>
  o <= add(x, UInt(1))
```

我们首先需要遍历AST到每个语句和表达式。然后，当我们看到一个指向字面量的DefNode时，我们需要将其存储到hashmap中并返回一个EmptyStmt（从而删除该DefNode）。然后，每当我们看到对已删除DefNode的引用时，必须插入相应的字面量。

```scala
对象 Inliner extends Pass {
  def name = "Inliner!"
  /** 在每个模块上运行inlineM **/
  def run(c: 电路): 电路 = c.copy(modules = c.modules map(inlineM(_)))

  /** 在每个模块的主体上运行inlineS **/
  def inlineM(m: DefModule): DefModule = m map inlineS(mutable.HashMap[String, Expression]())

  /** 在所有子表达式上运行inlineE，然后在子语句上运行inlineS。
    * 如果语句是包含字面量的DefNode，更新values并
    *   返回EmptyStmt；否则返回语句。*/
  def inlineS(values: mutable.HashMap[String, Expression])(s: Statement): Statement =
    s map inlineE(values) map inlineS(values) match {
      case d: DefNode => d.值 match {
        case l: 字面量 =>
          values(d.name) = l
          EmptyStmt
        case _ => d
      }
      case o => o 
    }

  /** 如果e是一个引用，且其名称包含在values中， 
    *   返回values(e.name)；否则在所有 
    *   子表达式上运行inlineE。*/
  def inlineE(values: mutable.HashMap[String, Expression])(e: Expression): Expression = e match {
    case e: Ref if values.contains(e.name) => values(e.name)
    case _ => e map inlineE(values)
  }
}
```

### 添加一个Primop
这个有用吗？通过向[firrtl仓库](https://github.com/freechipsproject/firrtl)提交问题让[@azidar](https://github.com/azidar)知道！

### 交换语句
这个有用吗？通过向[firrtl仓库](https://github.com/freechipsproject/firrtl)提交问题让[@azidar](https://github.com/azidar)知道！
